# Sentiment Analysis with PySpark

**Author:** Xiangrui Feng
**Model:** TF-IDF / HashingTF + Logistic Regression
**Datasets:** Steam, Social, Twitter (shared), ALL

## Introduction

The objective of this notebook is to perform sentiment analysis using PySpark.

We evaluate a Logistic Regression classifier on several datasets:

- Steam (our own dataset)
- Social dataset
- Shared normalized Twitter dataset
- Combined dataset (ALL)

The evaluation metrics are:

- Accuracy
- Precision
- Recall
- F1-score

In [1]:
import os
import sys

current_python = sys.executable

os.environ.setdefault("HADOOP_HOME", r"C:\hadoop")
os.environ.setdefault("hadoop.home.dir", r"C:\hadoop")
os.environ["PYSPARK_PYTHON"] = current_python
os.environ["PYSPARK_DRIVER_PYTHON"] = current_python

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SentimentAnalysisNotebook")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.default.parallelism", "64")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Spark started.")

Spark started.


In [2]:
DATASET = "all"   # "steam" / "social" / "twitter_shared" / "all"
USE_STOPWORDS = True
USE_IDF = True
NUM_FEATURES = 2**16
MAX_CHARS = 3000
MIN_CHARS = 5

In [3]:
if DATASET == "steam":
    TRAIN_CSV = "G:/Vmi/S2/BigData/SteamReview/prepared/train_dataset.csv"
    TEST_CSV  = "G:/Vmi/S2/BigData/SteamReview/prepared/test_dataset.csv"
    MODEL_DIR = "models/steam_sentiment_lr"

elif DATASET == "social":
    TRAIN_CSV = "G:/Vmi/S2/BigData/SteamReview/test/prepared/social_train.csv"
    TEST_CSV  = "G:/Vmi/S2/BigData/SteamReview/test/prepared/social_test.csv"
    MODEL_DIR = "models/social_sentiment_lr"

elif DATASET == "twitter_shared":
    TRAIN_CSV = "G:/Vmi/S2/BigData/SteamReview/test/prepared/standardized/twitter_train_32k.csv"
    TEST_CSV  = "G:/Vmi/S2/BigData/SteamReview/test/prepared/standardized/twitter_test_6.4k.csv"
    MODEL_DIR = "models/twitter_shared_sentiment_lr"

elif DATASET == "all":
    TRAIN_CSV = "G:/Vmi/S2/BigData/SteamReview/test/all_train.csv"
    TEST_CSV  = "G:/Vmi/S2/BigData/SteamReview/test/all_test.csv"
    MODEL_DIR = "models/all_sentiment_lr"

else:
    raise ValueError("Unknown dataset")

print("Dataset:", DATASET)
print("Train path:", TRAIN_CSV)
print("Test path :", TEST_CSV)
print("Model dir :", MODEL_DIR)

Dataset: all
Train path: G:/Vmi/S2/BigData/SteamReview/test/all_train.csv
Test path : G:/Vmi/S2/BigData/SteamReview/test/all_test.csv
Model dir : models/all_sentiment_lr


In [4]:
from pyspark.sql.functions import col

train_df = spark.read.csv(TRAIN_CSV, header=True, inferSchema=True)
test_df = spark.read.csv(TEST_CSV, header=True, inferSchema=True)

print("Train columns:", train_df.columns)
print("Test columns:", test_df.columns)

train_df.show(5, truncate=False)

Train columns: ['clean_text', 'sentiment_label']
Test columns: ['clean_text', 'sentiment_label']
+------------------------------------------------------------------------------------+---------------+
|clean_text                                                                          |sentiment_label|
+------------------------------------------------------------------------------------+---------------+
|Last day before I go back to work so I'll mainly be washing and ironing             |negative       |
|@stevenotwinery I tried to DM you back , but it won't let me - you don't follow me -|negative       |
|@DanceNsing iz SOO mean to me                                                       |negative       |
|ive never wanted to drag my nuts across a jaguar's front teeth so much until now    |negative       |
|WTF! ... Another day off. i need money!                                             |negative       |
+------------------------------------------------------------------------------

## Label Processing

The sentiment labels are converted into binary numerical labels:

- positive → 1.0
- negative → 0.0

In [5]:
from pyspark.sql.functions import when, lit

train_df = train_df.select("clean_text", "sentiment_label").filter(col("clean_text").isNotNull())
test_df = test_df.select("clean_text", "sentiment_label").filter(col("clean_text").isNotNull())

train_df = train_df.withColumn(
    "label",
    when(col("sentiment_label") == "positive", lit(1.0)).otherwise(lit(0.0))
)

test_df = test_df.withColumn(
    "label",
    when(col("sentiment_label") == "positive", lit(1.0)).otherwise(lit(0.0))
)

train_df.groupBy("sentiment_label").count().show()
test_df.groupBy("sentiment_label").count().show()

+--------------------+-----+
|     sentiment_label|count|
+--------------------+-----+
| and none of them...|    1|
| screwed these pu...|    1|
| it's 1"". I've p...|    1|
| but much more co...|    1|
| though I guess i...|    1|
| with no side eff...|    1|
| and the battery ...|    1|
| I DO NOT RECOMME...|    1|
| I did have 1 tub...|    1|
| as I want all 4 ...|    1|
| Hexagon Full Met...|    1|
| so don't fit exa...|    1|
| was extremely ea...|    1|
| only to find tha...|    1|
| powerful and sof...|    1|
| I find they are ...|    1|
| I wanted to cont...|    1|
|            positive|18609|
|       I can now see|    1|
|            negative|13359|
+--------------------+-----+
only showing top 20 rows

+--------------------+-----+
|     sentiment_label|count|
+--------------------+-----+
|              garden|    1|
| and would have l...|    1|
| it settled down ...|    1|
|         Powder Free|    1|
|            positive| 3769|
|            negative| 2627|
+----------------

## Text Cleaning

We remove empty or very short texts and truncate long texts to reduce memory usage.

In [6]:
from pyspark.sql.functions import length, substring

train_df = train_df.withColumn("clean_text", substring(col("clean_text"), 1, MAX_CHARS))
test_df = test_df.withColumn("clean_text", substring(col("clean_text"), 1, MAX_CHARS))

train_df = train_df.filter(length(col("clean_text")) >= MIN_CHARS)
test_df = test_df.filter(length(col("clean_text")) >= MIN_CHARS)

print(f"Train={train_df.count():,} | Test={test_df.count():,}")

Train=31,979 | Test=6,399


## Feature Engineering

We use the following text processing steps:

1. Tokenization
2. Stopword removal
3. HashingTF
4. Optional IDF weighting

In [7]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF

tokenizer = Tokenizer(inputCol="clean_text", outputCol="tokens_raw")
stages = [tokenizer]

if USE_STOPWORDS:
    remover = StopWordsRemover(inputCol="tokens_raw", outputCol="tokens")
    stages.append(remover)
    tokens_col = "tokens"
else:
    tokens_col = "tokens_raw"

tf = HashingTF(inputCol=tokens_col, outputCol="tf", numFeatures=NUM_FEATURES)
stages.append(tf)

if USE_IDF:
    idf = IDF(inputCol="tf", outputCol="features")
    stages.append(idf)
    feat_col = "features"
    method_name = "TF-IDF + LR"
else:
    feat_col = "tf"
    method_name = "HashingTF + LR"

print("Method:", method_name)

Method: TF-IDF + LR


In [8]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol=feat_col,
    labelCol="label",
    maxIter=30,
    regParam=0.1,
    elasticNetParam=0.0
)

In [9]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=stages + [lr])

train_df = train_df.select("clean_text", "label").cache()
test_df = test_df.select("clean_text", "label").cache()

model = pipeline.fit(train_df)
pred = model.transform(test_df)

print("Training completed.")

Training completed.


## Evaluation

We evaluate the model using:
- Accuracy
- Precision
- Recall
- F1-score

In [10]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)
evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision"
)
evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall"
)

acc = evaluator_acc.evaluate(pred)
f1 = evaluator_f1.evaluate(pred)
precision = evaluator_precision.evaluate(pred)
recall = evaluator_recall.evaluate(pred)

print(f"Method: {method_name}")
print(f"Accuracy  = {acc:.4f}")
print(f"Precision = {precision:.4f}")
print(f"Recall    = {recall:.4f}")
print(f"F1-score  = {f1:.4f}")

Method: TF-IDF + LR
Accuracy  = 0.6826
Precision = 0.6780
Recall    = 0.6826
F1-score  = 0.6777


In [11]:
pred.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 1419|
|  0.0|       1.0| 1212|
|  1.0|       0.0|  819|
|  1.0|       1.0| 2949|
+-----+----------+-----+



In [12]:
os.makedirs("models", exist_ok=True)

model.write().overwrite().save(MODEL_DIR)
print("Saved model to:", MODEL_DIR)

Saved model to: models/all_sentiment_lr


## Model Loading and Reuse

The trained pipeline can be saved and loaded later without retraining.

In [13]:
from pyspark.ml import PipelineModel

loaded_model = PipelineModel.load(MODEL_DIR)
print("Model loaded successfully.")

Model loaded successfully.


In [14]:
new_data = spark.createDataFrame([
    ("This game is amazing and very enjoyable",),
    ("Terrible experience, too many bugs",),
    ("The game is fine but could be better",)
], ["clean_text"])

new_pred = loaded_model.transform(new_data)
new_pred.select("clean_text", "prediction", "probability").show(truncate=False)

+---------------------------------------+----------+----------------------------------------+
|clean_text                             |prediction|probability                             |
+---------------------------------------+----------+----------------------------------------+
|This game is amazing and very enjoyable|1.0       |[0.22516792245370887,0.7748320775462911]|
|Terrible experience, too many bugs     |0.0       |[0.5080039856989362,0.49199601430106377]|
|The game is fine but could be better   |1.0       |[0.27630605988994317,0.7236939401100568]|
+---------------------------------------+----------+----------------------------------------+



In [15]:
pred_labeled = new_pred.withColumn(
    "predicted_sentiment",
    when(col("prediction") == 1.0, "positive").otherwise("negative")
)

pred_labeled.select("clean_text", "predicted_sentiment", "probability").show(truncate=False)

+---------------------------------------+-------------------+----------------------------------------+
|clean_text                             |predicted_sentiment|probability                             |
+---------------------------------------+-------------------+----------------------------------------+
|This game is amazing and very enjoyable|positive           |[0.22516792245370887,0.7748320775462911]|
|Terrible experience, too many bugs     |negative           |[0.5080039856989362,0.49199601430106377]|
|The game is fine but could be better   |positive           |[0.27630605988994317,0.7236939401100568]|
+---------------------------------------+-------------------+----------------------------------------+



## Conclusion

The experiments show that model performance strongly depends on the dataset.

- The Social dataset gives the lowest performance because of its limited size.
- The Steam dataset gives moderate performance.
- The shared normalized Twitter dataset gives the best performance, highlighting the importance of data quality and preprocessing.
- The combined dataset (ALL) increases diversity, but also introduces heterogeneity and noise.

These experiments confirm that both data quality and dataset size are essential for sentiment analysis performance.

In [ ]:
spark.stop()
print("Done.")